# 🧠 MLP - Multi-Layer Perceptron

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)                      # semente: faz seus resultados serem iguais aos do professor
plt.rcParams["figure.figsize"] = (6.5, 3.6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
print("Tudo pronto!")

## Parte 1 — O neurônio e o passo para frente (forward)

### 📖 Conceito
Um neurônio é um **gerente** que recebe opiniões (entradas `x`), dá uma **importância** a cada uma (pesos `w`), soma tudo, acrescenta um empurrãozinho fixo (o **bias** `b`) e passa o resultado por um **porteiro** (a função de **ativação**).

$$\text{saída} = \text{ativação}(x_1 w_1 + x_2 w_2 + \dots + b)$$

Fazer essa conta da entrada até a saída é o **passo para frente (forward pass)**.

### ⚙️ Implementação

In [ ]:
def sigmoide(z):
    # espreme qualquer número para ficar entre 0 e 1 (parece uma probabilidade)
    return 1 / (1 + np.exp(-z))

def relu(z):
    # zera o que é negativo; deixa o positivo passar
    return np.maximum(0, z)

def neuronio(x, w, b, ativacao=sigmoide):
    z = np.dot(x, w) + b        # soma ponderada + bias
    return ativacao(z)

# Os mesmos números do Exercício 2 da lista teórica
x = np.array([2.0, 3.0])
w = np.array([0.5, -1.0])
b = 1.0

print("Soma ponderada :", np.dot(x, w) + b)
print("Saída com ReLU     :", neuronio(x, w, b, relu))
print("Saída com Sigmóide :", round(float(neuronio(x, w, b, sigmoide)), 4))

In [ ]:
# Vamos ENXERGAR os dois porteiros
z = np.linspace(-6, 6, 200)
plt.plot(z, sigmoide(z), label="Sigmóide")
plt.plot(z, relu(z) / 6, label="ReLU (dividida por 6 para caber)")
plt.axhline(0, color="gray", lw=0.8); plt.axvline(0, color="gray", lw=0.8)
plt.title("Funções de ativação"); plt.xlabel("z (soma ponderada)"); plt.legend(); plt.show()

### ✅ Autocorreção

In [ ]:
assert np.isclose(np.dot(x, w) + b, -1.0), "A soma ponderada deveria ser -1"
assert neuronio(x, w, b, relu) == 0, "A ReLU deveria devolver 0 para um número negativo"
print("✔ Parte 1 ok! (Repare: a ReLU 'calou' o neurônio, exatamente como no Ex. 2)")

### 🛠️ Sua vez
1. Mude **apenas um peso** (`w`) para que a saída da ReLU deixe de ser 0.
2. Em quais valores de `z` a sigmóide fica "achatada" (quase 0 ou quase 1)? Guarde essa pergunta, ela volta na **Parte 9**.

In [ ]:
# escreva aqui

## Parte 2 — Função de perda

### 📖 Conceito
Para corrigir o erro, antes é preciso **medir** o erro com **um único número**. Usaremos duas notas de prova:

- **Erro quadrático médio (MSE)**: para prever números (preço de casa). Mede a distância ao quadrado.
- **Entropia cruzada binária (BCE)**: para respostas do tipo sim/não. É o "professor severo": **errar com muita confiança custa caro**.

### ⚙️ Implementação

In [ ]:
def perda_mse(y_prev, y_real):
    return np.mean((y_prev - y_real) ** 2)

def perda_bce(y_prev, y_real, eps=1e-12):
    y_prev = np.clip(y_prev, eps, 1 - eps)          # evita log(0)
    return -np.mean(y_real * np.log(y_prev) + (1 - y_real) * np.log(1 - y_prev))

# Exercício 4 da lista: preço real = 300 (mil). Rede A chutou 310, rede B chutou 500.
print("MSE rede A:", perda_mse(np.array([310.0]), np.array([300.0])))
print("MSE rede B:", perda_mse(np.array([500.0]), np.array([300.0])))

# Exercício 6 da lista: a imagem é de um CACHORRO (y_real = 0 para 'gato').
# X disse '99% gato'. Y disse '51% gato'.
cachorro = np.array([0.0])
print("\nBCE rede X (99% gato):", round(float(perda_bce(np.array([0.99]), cachorro)), 3))
print("BCE rede Y (51% gato):", round(float(perda_bce(np.array([0.51]), cachorro)), 3))

### ✅ Autocorreção

In [ ]:
assert perda_mse(np.array([500.0]), np.array([300.0])) > perda_mse(np.array([310.0]), np.array([300.0]))
assert perda_bce(np.array([0.99]), cachorro) > perda_bce(np.array([0.51]), cachorro)
print("✔ Parte 2 ok! Errar com confiança dói mais.")

### 🛠️ Sua vez
Gere um gráfico da perda BCE quando o gabarito é `0`, variando a previsão de 0.01 a 0.99. A curva sobe devagar ou explode perto de 1? O que isso diz sobre "confiança errada"?

In [ ]:
# escreva aqui

## Parte 3 — De quem é a culpa? Descobrindo o gradiente "cutucando" o peso

### 📖 Conceito
A rede errou. **Qual peso precisa girar, para qual lado, e quanto?** Há um jeito ridiculamente simples de descobrir, sem nenhuma derivada:

> **Cutuque o peso um tiquinho para cima e para baixo e veja como a perda reage.**

- Se a perda **sobe** quando o peso sobe → o peso deve **diminuir** (culpa positiva).
- Se a perda **desce** quando o peso sobe → o peso deve **aumentar** (culpa negativa).
- Se a perda quase não muda → o peso tem **pouca culpa**.

Esse "grau de culpa" é o **gradiente**. Aqui, vamos calculá-lo na marra, só cutucando.

### ⚙️ Implementação

In [ ]:
# Dados: porta lógica OR (saída 1 se pelo menos uma entrada for 1)
X_or = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_or = np.array([0, 1, 1, 1], dtype=float)

def perda_do_neuronio(w, b, X, y):
    previsao = sigmoide(X @ w + b)
    return perda_bce(previsao, y)

def culpa_cutucando(w, b, X, y, h=1e-5):
    # Para cada peso: cutuca pra cima, cutuca pra baixo, compara.
    culpa_w = np.zeros_like(w)
    for i in range(len(w)):
        w_cima  = w.copy(); w_cima[i]  += h
        w_baixo = w.copy(); w_baixo[i] -= h
        culpa_w[i] = (perda_do_neuronio(w_cima, b, X, y) - perda_do_neuronio(w_baixo, b, X, y)) / (2 * h)
    culpa_b = (perda_do_neuronio(w, b + h, X, y) - perda_do_neuronio(w, b - h, X, y)) / (2 * h)
    return culpa_w, culpa_b

w0 = np.array([0.5, -0.5]); b0 = 0.0
culpa_w, culpa_b = culpa_cutucando(w0, b0, X_or, y_or)
print("Perda atual:", round(float(perda_do_neuronio(w0, b0, X_or, y_or)), 4))
print("Culpa dos pesos:", culpa_w.round(4), "| culpa do bias:", round(float(culpa_b), 4))

**Como ler:** o primeiro peso tem culpa **negativa** (a perda cai se ele *aumentar*), então o ajuste vai **aumentá-lo**. O segundo tem culpa mais forte e também negativa: é o maior responsável pelo erro.

### ✅ Autocorreção

In [ ]:
# Prova do conceito: andar NO SENTIDO CONTRÁRIO da culpa tem que diminuir a perda
perda_antes  = perda_do_neuronio(w0, b0, X_or, y_or)
perda_depois = perda_do_neuronio(w0 - 0.1 * culpa_w, b0 - 0.1 * culpa_b, X_or, y_or)
print(f"Antes: {perda_antes:.4f}  →  Depois de um passinho: {perda_depois:.4f}")
assert perda_depois < perda_antes
print("✔ Parte 3 ok! Seguir a bússola da culpa diminui o erro.")

### 🛠️ Sua vez
1. Coloque `h = 1` (cutucada gigante) e depois `h = 1e-12` (cutucada microscópica). O resultado fica pior nos dois casos? Por quê?
2. Em uma rede com **1 milhão de pesos**, quantas vezes você teria de calcular a perda inteira usando este método? Guarde a resposta: é exatamente por isso que a retropropagação existe (Parte 4).

In [ ]:
# escreva aqui

## Parte 4 — Treinando um neurônio: as 4 etapas e a taxa de aprendizado

### 📖 Conceito
Cutucar peso por peso é lento demais. A **retropropagação** calcula a culpa de **todos os pesos de uma vez**, numa única passada para trás. Para um neurônio com sigmóide e perda BCE, a culpa fica incrivelmente simples:

> **culpa da saída = previsão − gabarito**

O ciclo de treino é sempre o mesmo, as **4 etapas**:

1. **Passo para frente:** calcula a previsão.
2. **Erro:** mede a perda.
3. **Culpa (retropropagação):** calcula o gradiente.
4. **Ajuste:** `peso novo = peso velho − taxa × culpa`.

### ⚙️ Implementação

In [ ]:
def treinar_neuronio(X, y, lr=0.5, epocas=500, seed=0):
    rng = np.random.default_rng(seed)
    w = rng.normal(0, 0.5, X.shape[1])       # pesos iniciais aleatórios pequenos (Ex. 14)
    b = 0.0
    historico = []
    for _ in range(epocas):
        previsao = sigmoide(X @ w + b)                 # 1) passo para frente
        historico.append(perda_bce(previsao, y))       # 2) erro
        culpa = previsao - y                           # 3) culpa na saída
        grad_w = X.T @ culpa / len(y)                  #    culpa de cada peso
        grad_b = culpa.mean()                          #    culpa do bias
        w = w - lr * grad_w                            # 4) ajuste dos pesos
        b = b - lr * grad_b
    return w, b, historico

w_or, b_or, hist = treinar_neuronio(X_or, y_or, lr=0.5, epocas=500)
plt.plot(hist); plt.title("Perda ao longo do treino (porta OR)")
plt.xlabel("época"); plt.ylabel("perda"); plt.show()
print("Previsões:", sigmoide(X_or @ w_or + b_or).round(3), "| gabarito:", y_or)

### ✅ Autocorreção — a retropropagação bate com o 'cutucar'?

In [ ]:
# Mesmo ponto da Parte 3: a fórmula pronta tem que dar o MESMO resultado que cutucar peso a peso
previsao = sigmoide(X_or @ w0 + b0)
culpa = previsao - y_or
grad_w_retro = X_or.T @ culpa / len(y_or)
grad_b_retro = culpa.mean()

print("Cutucando     :", culpa_w.round(6), round(float(culpa_b), 6))
print("Retropropagação:", grad_w_retro.round(6), round(float(grad_b_retro), 6))
assert np.allclose(culpa_w, grad_w_retro, atol=1e-6) and np.isclose(culpa_b, grad_b_retro, atol=1e-6)
assert hist[-1] < hist[0]
print("✔ Parte 4 ok! Mesmo resultado, mas a retropropagação precisa de UMA passada em vez de uma por peso.")

### 🛠️ Sua vez — taxa de aprendizado (Ex. 10 da lista teórica)
Na porta OR os dados são "fáceis" demais (qualquer taxa funciona). Para ver a taxa fazer diferença, vamos usar dois grupos de pontos que **se misturam um pouco**, como no mundo real. Rode a célula e responda: qual taxa é **lenta demais**? Qual **oscila**? Qual **explode**? Qual é a melhor?

In [ ]:
from sklearn.datasets import make_blobs
X_mist, y_mist = make_blobs(n_samples=200, centers=[[0, 0], [1.5, 1.5]], cluster_std=1.0, random_state=0)

for lr in [0.01, 0.1, 1, 10, 100]:
    _, _, h = treinar_neuronio(X_mist, y_mist.astype(float), lr=lr, epocas=300)
    plt.plot(h, label=f"lr = {lr}")
plt.yscale("log"); plt.title("Efeito da taxa de aprendizado"); plt.xlabel("época"); plt.ylabel("perda (escala log)")
plt.legend(); plt.show()

## Parte 5 — O limite de um neurônio: o problema do XOR

### 📖 Conceito
Um neurônio só consegue traçar **uma linha reta** para separar as classes. O **XOR** ("ou exclusivo": dá 1 quando as entradas são *diferentes*) tem os pontos 0 e 1 em **diagonais opostas**: nenhuma reta separa. Vamos provar isso treinando.

### ⚙️ Implementação

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=float)     # 0,1,1,0

w_x, b_x, hist_x = treinar_neuronio(X_xor, y_xor, lr=1.0, epocas=5000)
prev = sigmoide(X_xor @ w_x + b_x)
acc = np.mean((prev > 0.5) == y_xor)
print("Previsões :", prev.round(3))
print("Gabarito  :", y_xor)
print("Acurácia  :", acc)
plt.plot(hist_x); plt.title("Um neurônio tentando aprender XOR: a perda empaca")
plt.xlabel("época"); plt.ylabel("perda"); plt.show()

### ✅ Autocorreção

In [ ]:
assert acc <= 0.75, "Um único neurônio não deveria resolver o XOR!"
assert hist_x[-1] > 0.5, "A perda de um neurônio no XOR nunca deveria chegar perto de zero"
print("✔ Parte 5 ok! Mesmo com 5000 épocas, a retinha não resolve. Precisamos de uma camada oculta.")

### 🛠️ Sua vez
Aumente as épocas para 50.000 e a taxa para 5. Resolve? (Spoiler: o problema não é falta de treino, é falta de **capacidade**.)

In [ ]:
# escreva aqui

## Parte 6 — Rede de 2 camadas e a retropropagação "na unha"

### 📖 Conceito
Vamos montar uma rede **2 → n ocultos → 1**. Agora a culpa precisa **viajar de trás para frente**, como no telefone sem fio ao contrário:

1. Culpa na **saída**: `previsão − gabarito`.
2. Com essa culpa, achamos a culpa dos **pesos da última camada**.
3. A culpa é **repassada para a camada oculta**, passando pelos pesos (`@ W2.T`) e sendo multiplicada por "o quanto o neurônio estava ligado" (a derivada da ativação, `a1 * (1 - a1)` para a sigmóide).
4. Com a culpa da camada oculta, achamos a culpa dos **pesos da primeira camada**.

Repare que a culpa da camada oculta **reaproveita** a da saída. Esse reaproveitamento é o segredo da velocidade.

### ⚙️ Implementação, passo 6.1: criar os pesos aleatórios

In [ ]:
def iniciar_params(n_ocultos=4, seed=1):
    rng = np.random.default_rng(seed)
    return {
        "W1": rng.normal(0, 1, (2, n_ocultos)), "b1": np.zeros(n_ocultos),
        "W2": rng.normal(0, 1, (n_ocultos, 1)), "b2": np.zeros(1),
    }

params = iniciar_params()
print({k: v.shape for k, v in params.items()})

### ⚙️ Passo 6.2: o passo para frente (guardando tudo, pois a volta vai precisar!)

In [ ]:
def forward(X, p):
    z1 = X @ p["W1"] + p["b1"]       # soma ponderada da camada oculta
    a1 = sigmoide(z1)                # saída da camada oculta
    z2 = a1 @ p["W2"] + p["b2"]      # soma ponderada da camada de saída
    a2 = sigmoide(z2)                # previsão final
    return a2, (X, a1, a2)           # 'cache' guarda o que a volta precisa

y_col = y_xor.reshape(-1, 1)
prev, _ = forward(X_xor, params)
print("Perda inicial (rede ainda burra):", round(float(perda_bce(prev, y_col)), 4))

### ⚙️ Passo 6.3: o passo para trás (a retropropagação)

In [ ]:
def backward(y, p, cache):
    X, a1, a2 = cache
    N = len(y)

    culpa_saida = (a2 - y) / N                       # culpa na saída (previsão - gabarito)
    grad_W2 = a1.T @ culpa_saida                     # culpa dos pesos da última camada
    grad_b2 = culpa_saida.sum(axis=0)

    culpa_repassada = culpa_saida @ p["W2"].T        # culpa viaja de trás pra frente, pelos pesos
    culpa_oculta = culpa_repassada * a1 * (1 - a1)   # x "quão ligado" o neurônio estava
    grad_W1 = X.T @ culpa_oculta                     # culpa dos pesos da primeira camada
    grad_b1 = culpa_oculta.sum(axis=0)

    return {"W1": grad_W1, "b1": grad_b1, "W2": grad_W2, "b2": grad_b2}

prev, cache = forward(X_xor, params)
grads = backward(y_col, params, cache)
print({k: v.shape for k, v in grads.items()})

### ✅ Autocorreção — a "prova real" (gradient check)
O melhor teste de uma retropropagação escrita à mão é comparar com o método de cutucar da Parte 3. Se baterem, sua implementação está certa. **Todo mundo que escreve backprop faz isso.**

In [ ]:
def checar_gradiente(y, p, grads, h=1e-5):
    maior_diferenca = 0
    for nome in p:
        achatado = p[nome].reshape(-1)                      # visão do mesmo array
        for i in range(achatado.size):
            original = achatado[i]
            achatado[i] = original + h; l_cima  = perda_bce(forward(X_xor, p)[0], y)
            achatado[i] = original - h; l_baixo = perda_bce(forward(X_xor, p)[0], y)
            achatado[i] = original                          # devolve o peso ao lugar
            cutucada = (l_cima - l_baixo) / (2 * h)
            maior_diferenca = max(maior_diferenca, abs(cutucada - grads[nome].reshape(-1)[i]))
    return maior_diferenca

dif = checar_gradiente(y_col, params, grads)
print("Maior diferença entre 'cutucar' e retropropagação:", f"{dif:.2e}")
assert dif < 1e-6
print("✔ Parte 6 ok! Sua retropropagação está correta.")

### 🛠️ Sua vez
**Sabote o código de propósito:** na linha `culpa_oculta = ...`, apague o termo `* a1 * (1 - a1)`, rode de novo a autocorreção e veja o `assert` gritar. É assim que se descobre bug de backprop na vida real.

In [ ]:
# escreva aqui

## Parte 7 — Treinando a rede no XOR

### 📖 Conceito
Agora juntamos as 4 etapas em um laço: **frente → erro → culpa → ajuste**, repetido milhares de vezes. Se a camada oculta funcionar como esperado, o XOR cai.

### ⚙️ Implementação

In [ ]:
def treinar_xor(n_ocultos=4, lr=1.0, epocas=5000, seed=1):
    p = iniciar_params(n_ocultos, seed)
    historico = []
    for _ in range(epocas):
        prev, cache = forward(X_xor, p)                 # 1) frente
        historico.append(perda_bce(prev, y_col))        # 2) erro
        grads = backward(y_col, p, cache)               # 3) culpa
        for nome in p:                                  # 4) ajuste
            p[nome] -= lr * grads[nome]
    return p, historico

params_xor, hist_xor = treinar_xor()
prev, _ = forward(X_xor, params_xor)
print("Previsões:", prev.ravel().round(3), "| gabarito:", y_xor)
print("Acurácia :", np.mean((prev.ravel() > 0.5) == y_xor))

plt.plot(hist_xor); plt.title("Rede 2-4-1 aprendendo XOR")
plt.xlabel("época"); plt.ylabel("perda"); plt.show()

In [ ]:
def plotar_fronteira(funcao_prever, X, y, titulo=""):
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - .5, X[:, 0].max() + .5, 200),
                         np.linspace(X[:, 1].min() - .5, X[:, 1].max() + .5, 200))
    zz = funcao_prever(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.contourf(xx, yy, zz, levels=20, alpha=.55, cmap="coolwarm")
    plt.scatter(X[:, 0], X[:, 1], c=np.ravel(y), cmap="coolwarm", edgecolors="k", s=70)
    plt.title(titulo); plt.show()

plotar_fronteira(lambda G: forward(G, params_xor)[0], X_xor, y_xor, "Fronteira de decisão aprendida (não é uma reta!)")

### ✅ Autocorreção

In [ ]:
assert np.all((prev.ravel() > 0.5) == y_xor), "A rede deveria acertar os 4 casos do XOR"
assert hist_xor[-1] < 0.05
print("✔ Parte 7 ok! A camada oculta transformou o impossível em possível.")

### 🛠️ Sua vez
Treine com `n_ocultos = 1`, `2` e `8`. Quantos neurônios ocultos são o mínimo para resolver o XOR? A rede com 1 neurônio oculto lembra qual modelo da Parte 5?

In [ ]:
# escreva aqui (dica: treinar_xor(n_ocultos=1))

## Parte 8 — Rede geral, mini-batch, overfitting e parada antecipada

### 📖 Conceito
Chega de rede de tamanho fixo. Vamos criar uma **classe** que aceita quantas camadas você quiser e a ativação que preferir (`sigmoide`, `relu` ou `tanh`). Também vamos treinar com **mini-batches** e vigiar a **validação**, a "prova surpresa" que mostra se a rede decorou (overfitting) ou entendeu.

- **Época** = uma volta completa pelos dados de treino.
- **Iteração** = um ajuste de pesos (um mini-batch).

### ⚙️ Implementação: a classe `RedeNeural`

In [ ]:
class RedeNeural:
    def __init__(self, camadas, ativacao="relu", seed=0):
        # camadas = [entradas, ocultas..., 1]. Ex.: [2, 16, 16, 1]
        rng = np.random.default_rng(seed)
        self.ativacao = ativacao
        self.W, self.b = [], []
        for n_in, n_out in zip(camadas[:-1], camadas[1:]):
            # pesos aleatórios PEQUENOS, com escala que depende do tamanho da camada (Ex. 14)
            escala = np.sqrt(2 / n_in) if ativacao == "relu" else np.sqrt(1 / n_in)
            self.W.append(rng.normal(0, escala, (n_in, n_out)))
            self.b.append(np.zeros(n_out))

    # --- ativação e "quão ligado o neurônio estava" ---
    def _f(self, z):
        if self.ativacao == "relu": return np.maximum(0, z)
        if self.ativacao == "tanh": return np.tanh(z)
        return sigmoide(z)

    def _quao_ligado(self, z, a):
        if self.ativacao == "relu": return (z > 0).astype(float)
        if self.ativacao == "tanh": return 1 - a ** 2
        return a * (1 - a)

    # --- passo para frente: guarda tudo para a volta ---
    def forward(self, X):
        self.Z, self.A = [], [X]
        a = X
        ultima = len(self.W) - 1
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            z = a @ W + b
            a = sigmoide(z) if i == ultima else self._f(z)     # saída sempre sigmóide (0 a 1)
            self.Z.append(z); self.A.append(a)
        return a

    # --- passo para trás: a retropropagação para QUALQUER número de camadas ---
    def backward(self, y):
        N = len(y)
        culpa = (self.A[-1] - y) / N                           # culpa na saída
        gW = [None] * len(self.W); gb = [None] * len(self.W)
        for i in reversed(range(len(self.W))):
            gW[i] = self.A[i].T @ culpa                        # culpa dos pesos da camada i
            gb[i] = culpa.sum(axis=0)
            if i > 0:
                repassada = culpa @ self.W[i].T                # culpa viaja para a camada anterior
                culpa = repassada * self._quao_ligado(self.Z[i - 1], self.A[i])
        return gW, gb

    def prever_proba(self, X): return self.forward(X)
    def prever(self, X): return (self.forward(X) > 0.5).astype(int)

### ⚙️ A função de treino (mini-batch, validação e parada antecipada)

In [ ]:
def treinar(rede, Xtr, ytr, Xval=None, yval=None, epocas=100, lr=0.1, lote=32,
            paciencia=None, seed=0):
    rng = np.random.default_rng(seed)
    ytr = ytr.reshape(-1, 1)
    if yval is not None: yval = yval.reshape(-1, 1)
    hist = {"perda_tr": [], "perda_val": [], "acc_tr": [], "acc_val": []}
    melhor_val, melhores_pesos, sem_melhora = np.inf, None, 0

    for epoca in range(epocas):
        ordem = rng.permutation(len(Xtr))                     # embaralha a cada época
        for ini in range(0, len(Xtr), lote):
            idx = ordem[ini:ini + lote]
            rede.forward(Xtr[idx])                            # 1) frente
            gW, gb = rede.backward(ytr[idx])                  # 3) culpa (a perda é implícita na fórmula)
            for i in range(len(rede.W)):                      # 4) ajuste
                rede.W[i] -= lr * gW[i]
                rede.b[i] -= lr * gb[i]

        # --- boletim da época ---
        p_tr = rede.forward(Xtr)
        hist["perda_tr"].append(perda_bce(p_tr, ytr)); hist["acc_tr"].append(np.mean((p_tr > .5) == ytr))
        if Xval is not None:
            p_v = rede.forward(Xval)
            perda_v = perda_bce(p_v, yval)
            hist["perda_val"].append(perda_v); hist["acc_val"].append(np.mean((p_v > .5) == yval))
            if paciencia is not None:                         # parada antecipada
                if perda_v < melhor_val - 1e-6:
                    melhor_val, sem_melhora = perda_v, 0
                    melhores_pesos = ([W.copy() for W in rede.W], [b.copy() for b in rede.b])
                else:
                    sem_melhora += 1
                    if sem_melhora >= paciencia:
                        rede.W, rede.b = melhores_pesos       # volta para o melhor momento
                        hist["parou_em"] = epoca + 1
                        break
    return hist

### ✅ Autocorreção 1 — a classe também passa na 'prova real'

In [ ]:
rede_teste = RedeNeural([2, 5, 3, 1], ativacao="tanh", seed=3)
Xt, yt = X_xor, y_col
rede_teste.forward(Xt); gW, gb = rede_teste.backward(yt)

maior = 0
for lista_p, lista_g in [(rede_teste.W, gW), (rede_teste.b, gb)]:
    for P, G in zip(lista_p, lista_g):
        achatado = P.reshape(-1)
        for i in range(achatado.size):
            o = achatado[i]
            achatado[i] = o + 1e-5; l1 = perda_bce(rede_teste.forward(Xt), yt)
            achatado[i] = o - 1e-5; l2 = perda_bce(rede_teste.forward(Xt), yt)
            achatado[i] = o
            maior = max(maior, abs((l1 - l2) / 2e-5 - G.reshape(-1)[i]))
print("Maior diferença (rede com 3 camadas, tanh):", f"{maior:.2e}")
assert maior < 1e-6
print("✔ A retropropagação genérica está correta para qualquer número de camadas.")

### ⚙️ Experimento: dados em formato de duas luas (mais difícil que o XOR)
Vamos criar propositalmente **poucos dados de treino** e uma rede **gigante**, para provocar overfitting.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X_luas, y_luas = make_moons(n_samples=500, noise=0.35, random_state=0)
Xtr, Xval, ytr, yval = train_test_split(X_luas, y_luas, train_size=0.1, random_state=0)  # só 50 para treinar!
print("Treino:", Xtr.shape, "| Validação:", Xval.shape)

rede_grande = RedeNeural([2, 128, 128, 1], ativacao="relu", seed=0)
hist_grande = treinar(rede_grande, Xtr, ytr, Xval, yval, epocas=600, lr=0.1, lote=8)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(hist_grande["perda_tr"], label="treino"); ax[0].plot(hist_grande["perda_val"], label="validação")
ax[0].set_title("Perda: a validação sobe = overfitting"); ax[0].set_xlabel("época"); ax[0].legend()
ax[1].plot(hist_grande["acc_tr"], label="treino"); ax[1].plot(hist_grande["acc_val"], label="validação")
ax[1].set_title("Acurácia"); ax[1].set_xlabel("época"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"Acurácia final → treino: {hist_grande['acc_tr'][-1]:.2f} | validação: {hist_grande['acc_val'][-1]:.2f}")

### ⚙️ O remédio: parada antecipada (*early stopping*)

In [ ]:
rede_es = RedeNeural([2, 128, 128, 1], ativacao="relu", seed=0)
hist_es = treinar(rede_es, Xtr, ytr, Xval, yval, epocas=600, lr=0.1, lote=8, paciencia=20)
print("Treino parou na época:", hist_es.get("parou_em", "não parou (rodou tudo)"))
print(f"Perda de validação → sem parada antecipada: {hist_grande['perda_val'][-1]:.3f} | com: {perda_bce(rede_es.forward(Xval), yval.reshape(-1,1)):.3f}")
plotar_fronteira(lambda G: rede_es.forward(G), Xval, yval, "Rede com parada antecipada (dados de validação)")

### ✅ Autocorreção 2 — contas de época e iteração (Ex. 11 da lista teórica)

In [ ]:
import math
n_treino, tam_lote = 1000, 100
iteracoes_por_epoca = math.ceil(n_treino / tam_lote)
print("Iterações por época:", iteracoes_por_epoca, "| em 20 épocas:", iteracoes_por_epoca * 20)
assert iteracoes_por_epoca == 10 and iteracoes_por_epoca * 20 == 200
assert hist_grande["perda_val"][-1] > 2 * min(hist_grande["perda_val"]), "A perda de validação deveria ter disparado depois de um ponto (overfitting)"
assert hist_grande["acc_tr"][-1] - hist_grande["acc_val"][-1] > 0.1, "Treino deveria estar bem melhor que a validação"
print("✔ Parte 8 ok!")

### 🛠️ Sua vez
1. Repita o treino com `lote=len(Xtr)` (batch completo) e com `lote=1` (estocástico). Compare a **suavidade** da curva e o **tempo** (use `%%time` no topo da célula).
2. Reduza a rede para `[2, 8, 1]`. O overfitting some? E se mantiver `[2, 128, 128, 1]` mas treinar com `train_size=0.5` (250 pontos)?
3. Troque `ativacao="relu"` por `"sigmoide"` e `"tanh"`. Qual converge mais rápido?

In [ ]:
# escreva aqui

## Parte 9 — O gradiente que desaparece (vanishing gradient)

### 📖 Conceito
Na volta, a culpa é **multiplicada** a cada camada que atravessa. A sigmóide só deixa passar **no máximo 25%** da culpa em cada camada (lembra do 'telefone sem fio'?). Depois de 10 camadas, o que sobra é um sussurro. Já a ReLU repassa a culpa inteira nos neurônios ligados.

Vamos medir: criamos uma rede com 10 camadas ocultas e olhamos o **tamanho médio da culpa** em cada camada.

### ⚙️ Implementação

In [ ]:
def tamanho_da_culpa_por_camada(ativacao, camadas_ocultas=10, largura=16, seed=0):
    rng = np.random.default_rng(seed)
    rede = RedeNeural([10] + [largura] * camadas_ocultas + [1], ativacao=ativacao, seed=seed)
    X = rng.normal(size=(128, 10)); y = rng.integers(0, 2, size=(128, 1)).astype(float)
    rede.forward(X)
    gW, _ = rede.backward(y)
    return np.array([np.abs(g).mean() for g in gW])        # tamanho médio da culpa, camada a camada

culpa_sig  = tamanho_da_culpa_por_camada("sigmoide")
culpa_relu = tamanho_da_culpa_por_camada("relu")

camadas = np.arange(1, len(culpa_sig) + 1)
plt.semilogy(camadas, culpa_sig,  "o-", label="Sigmóide")
plt.semilogy(camadas, culpa_relu, "s-", label="ReLU")
plt.xlabel("camada (1 = a mais próxima da entrada)"); plt.ylabel("tamanho médio da culpa (escala log)")
plt.title("Quanta culpa chega em cada camada?"); plt.legend(); plt.show()

print(f"Sigmóide: culpa na 1ª camada / culpa na última = {culpa_sig[0] / culpa_sig[-1]:.1e}")
print(f"ReLU    : culpa na 1ª camada / culpa na última = {culpa_relu[0] / culpa_relu[-1]:.1e}")

### ✅ Autocorreção

In [ ]:
assert culpa_sig[0] / culpa_sig[-1] < 0.05, "Na sigmóide, a 1ª camada deveria receber uma fração minúscula da culpa"
assert culpa_relu[0] / culpa_relu[-1] > culpa_sig[0] / culpa_sig[-1] * 10, "A ReLU deveria preservar bem mais culpa"
print("✔ Parte 9 ok! A sigmóide em rede profunda deixa as primeiras camadas 'surdas'.")

### 🛠️ Sua vez
1. Mude `camadas_ocultas` para 3, 6, 10 e 20 e veja a razão da sigmóide despencar.
2. Treine uma rede de 8 camadas ocultas de sigmóide e outra de ReLU nos dados das luas (Parte 8). Qual aprende? Isso é o que justificou a popularização da ReLU.

In [ ]:
# escreva aqui

## Parte 10 — O que as bibliotecas fazem por você (PyTorch)

### 📖 Conceito
Nas Partes 1–9, **você** escreveu o `forward` e o `backward`. No PyTorch (e Keras), a retropropagação vem pronta como `perda.backward()`: ela é calculada automaticamente (*autograd*). As 4 etapas viram 5 linhas:

| Etapa | Em PyTorch |
| --- | --- |
| zerar a culpa antiga | `otimizador.zero_grad()` |
| 1) passo para frente | `saida = modelo(x)` |
| 2) erro | `perda = funcao_perda(saida, y)` |
| 3) culpa (retropropagação) | `perda.backward()` |
| 4) ajuste dos pesos | `otimizador.step()` |

### ⚙️ Implementação — a prova de que o `.backward()` faz o MESMO que o nosso

In [ ]:
import torch
import torch.nn as nn

# Pegamos os pesos da NOSSA rede e colocamos num modelo do PyTorch
nossa = RedeNeural([2, 5, 3, 1], ativacao="tanh", seed=3)
modelo = nn.Sequential(nn.Linear(2, 5), nn.Tanh(), nn.Linear(5, 3), nn.Tanh(), nn.Linear(3, 1), nn.Sigmoid())
camadas_lineares = [m for m in modelo if isinstance(m, nn.Linear)]
with torch.no_grad():
    for lin, W, b in zip(camadas_lineares, nossa.W, nossa.b):
        lin.weight.copy_(torch.tensor(W.T)); lin.bias.copy_(torch.tensor(b))

xt = torch.tensor(X_xor, dtype=torch.float64); yt_t = torch.tensor(y_col, dtype=torch.float64)
modelo = modelo.double()

perda_torch = nn.BCELoss()(modelo(xt), yt_t)
perda_torch.backward()                                    # <<< a retropropagação automática

nossa.forward(X_xor); gW_nossa, _ = nossa.backward(y_col)
diferenca = max(np.abs(gW_nossa[i] - lin.weight.grad.numpy().T).max() for i, lin in enumerate(camadas_lineares))
print("Perda PyTorch:", round(perda_torch.item(), 6), "| perda nossa:", round(float(perda_bce(nossa.forward(X_xor), y_col)), 6))
print("Maior diferença entre as culpas (nossa x PyTorch):", f"{diferenca:.2e}")

### ✅ Autocorreção

In [ ]:
assert diferenca < 1e-7
print("✔ O .backward() do PyTorch calcula exatamente a mesma culpa que você programou na unha!")

### ⚙️ O laço de treino completo em PyTorch (duas luas)

In [ ]:
torch.manual_seed(0)
Xtr_t  = torch.tensor(Xtr,  dtype=torch.float32); ytr_t  = torch.tensor(ytr,  dtype=torch.float32).view(-1, 1)
Xval_t = torch.tensor(Xval, dtype=torch.float32); yval_t = torch.tensor(yval, dtype=torch.float32).view(-1, 1)

modelo = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 16), nn.ReLU(), nn.Linear(16, 1))
funcao_perda = nn.BCEWithLogitsLoss()                      # sigmóide + BCE juntas (mais estável)
otimizador = torch.optim.Adam(modelo.parameters(), lr=0.01)   # troque por torch.optim.SGD(..., lr=0.05) e compare (Ex. 17)

perdas_tr, perdas_val = [], []
for epoca in range(150):
    ordem = torch.randperm(len(Xtr_t))
    for ini in range(0, len(Xtr_t), 16):
        idx = ordem[ini:ini + 16]
        otimizador.zero_grad()                             # zera a culpa antiga (Ex. 19: o bug do Trecho B)
        saida = modelo(Xtr_t[idx])                         # 1) passo para frente
        perda = funcao_perda(saida, ytr_t[idx])            # 2) erro
        perda.backward()                                   # 3) culpa (retropropagação)
        otimizador.step()                                  # 4) ajuste dos pesos
    with torch.no_grad():
        perdas_tr.append(funcao_perda(modelo(Xtr_t), ytr_t).item())
        perdas_val.append(funcao_perda(modelo(Xval_t), yval_t).item())

acc_val = ((modelo(Xval_t) > 0).float() == yval_t).float().mean().item()
plt.plot(perdas_tr, label="treino"); plt.plot(perdas_val, label="validação")
plt.title("PyTorch + Adam nas duas luas"); plt.xlabel("época"); plt.legend(); plt.show()
print(f"Acurácia de validação: {acc_val:.2f}")

### 🛠️ Sua vez
1. Troque `Adam` por `SGD` (com `lr=0.05`) e compare as curvas. Qual desce mais rápido? (Ex. 17)
2. **Sabote:** comente a linha `otimizador.zero_grad()` e observe o que acontece com a perda. Você acabou de reproduzir o bug do Trecho B do Ex. 19.
3. **Sabote:** comente `otimizador.step()`. A perda se mexe? (Trecho A do Ex. 19.)

In [ ]:
# escreva aqui

## Parte 11 — 🏆 Desafio final: um problema real do Kaggle

### O problema: **Spaceship Titanic**
🔗 https://www.kaggle.com/competitions/spaceship-titanic

> Em 2912, a nave *Spaceship Titanic* colidiu com uma anomalia espaço-temporal e **quase metade dos passageiros foi transportada para outra dimensão**. Sua missão: usando os dados dos passageiros, **prever quem foi transportado** (`Transported`: `True`/`False`).

| Item | Detalhe |
| --- | --- |
| Tipo de problema | Classificação binária (igual à nossa BCE!) |
| Métrica oficial | Acurácia (% de passageiros classificados corretamente) |
| Dados | `train.csv` (~8.700 passageiros, com o gabarito) e `test.csv` (~4.300, sem o gabarito) |
| Colunas | `PassengerId`, `HomePlanet`, `CryoSleep`, `Cabin`, `Destination`, `Age`, `VIP`, `RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck`, `Name` |
| Por que é ótimo para a aula | Tem **valores ausentes**, **categorias**, números em **escalas bem diferentes** e uma coluna composta (`Cabin`): é exatamente o pré-processamento do 1º bimestre alimentando a rede do 2º. |

> ⚠️ Os dados vêm do site do Kaggle. Baixe `train.csv` e `test.csv`.No Colab: faça upload dos arquivos pelo painel de arquivos.

### Roteiro (cada passo ligado a um conceito da lista)
1. **Explorar** os dados (valores ausentes, tipos).
2. **Pré-processar:** preencher ausentes, separar `Cabin`, one-hot encoding, **normalizar** (Ex. 16).
3. **Separar treino/validação** (Ex. 15).
4. **Montar a rede** e **treinar** com o laço das 4 etapas (Ex. 9, 19).
5. **Diagnosticar** a curva de perda (Ex. 18) e **ajustar hiperparâmetros** (Ex. 10, 11, 16, 17).
6. **Gerar o `submission.csv`** e subir no Kaggle.

### Passo 1 — Carregar e explorar

In [ ]:
import pandas as pd
from pathlib import Path

PASTA = Path("spaceship-titanic")
assert (PASTA / "train.csv").exists() and (PASTA / "test.csv").exists(), \
    "Baixe train.csv e test.csv em kaggle.com/competitions/spaceship-titanic e coloque na pasta 'spaceship-titanic/'"

treino = pd.read_csv(PASTA / "train.csv")
teste  = pd.read_csv(PASTA / "test.csv")
print("Treino:", treino.shape, "| Teste:", teste.shape)
display(treino.head())
print("\nValores ausentes por coluna (treino):")
print(treino.isna().sum().sort_values(ascending=False))
print("\nProporção de transportados:", treino["Transported"].mean().round(3))

### Passo 2 — Pré-processamento (cuidado: tudo que 'aprende' algo dos dados usa só o TREINO)

In [ ]:
GASTOS = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

def engenharia_de_atributos(df):
    df = df.copy()
    # 'Cabin' vem como 'B/0/P' = deck / número / lado
    cab = df["Cabin"].str.split("/", expand=True)
    df["Deck"], df["CabNum"], df["Lado"] = cab[0], pd.to_numeric(cab[1], errors="coerce"), cab[2]
    # grupo de viagem vem em 'PassengerId' = 'gggg_pp'
    df["Grupo"] = df["PassengerId"].str.split("_").str[0]
    df["TamGrupo"] = df.groupby("Grupo")["PassengerId"].transform("count")
    df["GastoTotal"] = df[GASTOS].sum(axis=1)
    df["CryoSleep"] = df["CryoSleep"].astype("object")
    df["VIP"] = df["VIP"].astype("object")
    return df

tr = engenharia_de_atributos(treino)
te = engenharia_de_atributos(teste)

NUMERICAS   = ["Age", "CabNum", "TamGrupo", "GastoTotal"] + GASTOS
CATEGORICAS = ["HomePlanet", "CryoSleep", "Destination", "VIP", "Deck", "Lado"]

# 1) preencher ausentes com a mediana / moda DO TREINO
for c in NUMERICAS:
    tr[c] = tr[c].fillna(tr[c].median()); te[c] = te[c].fillna(tr[c].median())
for c in CATEGORICAS:
    moda = tr[c].mode()[0]
    tr[c] = tr[c].fillna(moda).astype(str); te[c] = te[c].fillna(moda).astype(str)

# 2) gastos têm cauda enorme: log(1 + x) comprime os valores gigantes
for c in GASTOS + ["GastoTotal"]:
    tr[c] = np.log1p(tr[c]); te[c] = np.log1p(te[c])

# 3) one-hot das categorias (juntando treino+teste só para ter as mesmas colunas)
todas = pd.concat([tr[CATEGORICAS], te[CATEGORICAS]], keys=["tr", "te"])
todas = pd.get_dummies(todas, dtype=float)
OH_tr, OH_te = todas.loc["tr"].values, todas.loc["te"].values

# 4) padronização (z-score) com média e desvio DO TREINO
media, desvio = tr[NUMERICAS].mean(), tr[NUMERICAS].std().replace(0, 1)
NUM_tr = ((tr[NUMERICAS] - media) / desvio).values
NUM_te = ((te[NUMERICAS] - media) / desvio).values

X_all = np.hstack([NUM_tr, OH_tr]).astype("float32")
X_teste = np.hstack([NUM_te, OH_te]).astype("float32")
y_all = treino["Transported"].astype(int).values.astype("float32")
print("Matriz de entrada:", X_all.shape, "| Teste:", X_teste.shape)

### Passo 3 — Separar treino e validação

In [ ]:
from sklearn.model_selection import train_test_split
Xtr_k, Xval_k, ytr_k, yval_k = train_test_split(X_all, y_all, test_size=0.2, random_state=0, stratify=y_all)
print("Treino:", Xtr_k.shape, "| Validação:", Xval_k.shape)

### Passo 4 — Montar a rede e treinar (as 4 etapas, versão PyTorch)

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)

Xtr_t, ytr_t = torch.tensor(Xtr_k), torch.tensor(ytr_k).view(-1, 1)
Xval_t, yval_t = torch.tensor(Xval_k), torch.tensor(yval_k).view(-1, 1)

# ----- HIPERPARÂMETROS (a parte que você vai ajustar!) -----
OCULTAS, TAXA, LOTE, EPOCAS, PACIENCIA = [64, 32], 1e-3, 64, 100, 10
# -----------------------------------------------------------

camadas, n_in = [], Xtr_t.shape[1]
for n in OCULTAS:
    camadas += [nn.Linear(n_in, n), nn.ReLU(), nn.Dropout(0.2)]      # Dropout: "desliga" neurônios (combate overfitting)
    n_in = n
camadas.append(nn.Linear(n_in, 1))
modelo = nn.Sequential(*camadas)

funcao_perda = nn.BCEWithLogitsLoss()
otimizador = torch.optim.Adam(modelo.parameters(), lr=TAXA, weight_decay=1e-4)

h = {"tr": [], "val": [], "acc_val": []}
melhor, melhor_estado, sem_melhora = np.inf, None, 0
for epoca in range(EPOCAS):
    modelo.train()
    ordem = torch.randperm(len(Xtr_t))
    for ini in range(0, len(Xtr_t), LOTE):
        idx = ordem[ini:ini + LOTE]
        otimizador.zero_grad()
        perda = funcao_perda(modelo(Xtr_t[idx]), ytr_t[idx])
        perda.backward()
        otimizador.step()
    modelo.eval()
    with torch.no_grad():
        h["tr"].append(funcao_perda(modelo(Xtr_t), ytr_t).item())
        pv = funcao_perda(modelo(Xval_t), yval_t).item()
        h["val"].append(pv)
        h["acc_val"].append(((modelo(Xval_t) > 0).float() == yval_t).float().mean().item())
    if pv < melhor - 1e-4:
        melhor, sem_melhora = pv, 0
        melhor_estado = {k: v.clone() for k, v in modelo.state_dict().items()}
    else:
        sem_melhora += 1
        if sem_melhora >= PACIENCIA:
            print(f"Parada antecipada na época {epoca + 1}")
            break

modelo.load_state_dict(melhor_estado)
print(f"Melhor perda de validação: {melhor:.4f} | acurácia de validação (última época): {h['acc_val'][-1]:.4f}")

### Passo 5 — Diagnóstico (volte à tabela de sintomas do Ex. 18)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(h["tr"], label="treino"); ax[0].plot(h["val"], label="validação")
ax[0].set_title("Perda"); ax[0].set_xlabel("época"); ax[0].legend()
ax[1].plot(h["acc_val"]); ax[1].set_title("Acurácia de validação"); ax[1].set_xlabel("época")
plt.tight_layout(); plt.show()

modelo.eval()
with torch.no_grad():
    acc_final = ((modelo(Xval_t) > 0).float() == yval_t).float().mean().item()
print(f"Acurácia de validação do melhor modelo: {acc_final:.4f}")

### 🛠️ Missões do desafio (do mais fácil ao mais difícil)
Anote a **acurácia de validação** a cada mudança, como num diário de bordo. Mude **uma coisa por vez**.

| # | Missão | Conceito |
| --- | --- | --- |
| 1 | Troque `TAXA` por 1e-1 e por 1e-5. Descreva as curvas. | Taxa de aprendizado (Ex. 10) |
| 2 | Teste `LOTE` = 16, 64 e 1024. Compare tempo e estabilidade. | Mini-batch (Ex. 11) |
| 3 | Mude `OCULTAS` para `[8]`, `[128, 64, 32]` e `[256, 256, 256]`. Quando surge overfitting? | Capacidade (Ex. 3, 15) |
| 4 | **Remova a padronização** (use `NUM_tr` sem dividir pelo desvio). O que acontece? | Normalização (Ex. 16) |
| 5 | Troque `Adam` por `SGD(lr=0.01)` e depois `SGD(lr=0.01, momentum=0.9)`. | Otimizadores (Ex. 17) |
| 6 | Troque `ReLU` por `Sigmoid` com 6 camadas ocultas. Houve gradiente sumindo? | Ex. 13 |
| 7 | Tire o `Dropout` e o `weight_decay`. Compare as curvas de treino e validação. | Regularização (Ex. 15) |
| 8 | **Chefão:** crie novos atributos (ex.: gasto por categoria, deck × lado, idade em faixas) e supere sua melhor acurácia. | Engenharia de atributos |

> 📈 **Referência de bolso:** baixar o dataset e acertar entre ~79% e ~81% de acurácia é um resultado muito razoável para uma rede simples. Se passar disso, você está indo muito bem. Compare com o placar público do Kaggle.

### 🎒 Outros problemas do Kaggle para continuar treinando

| Problema | Tipo | Por que treinar |
| --- | --- | --- |
| [Digit Recognizer (MNIST)](https://www.kaggle.com/competitions/digit-recognizer) | Imagens, 10 classes | O "Olá, mundo" das redes: uma rede densa já passa dos 97% |
| [Titanic](https://www.kaggle.com/competitions/titanic) | Tabular, binária | Versão menor deste desafio, ótima para aquecer |
| [House Prices](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques) | Tabular, **regressão** | Troque a BCE pela MSE (Parte 2) e tire a sigmóide da saída |

---
## 🏁 Resumo final: o que você aprendeu fazendo

1. Treinar = **girar pesos** até a perda cair.
2. **Gradiente = culpa** de cada peso; dá para estimar cutucando (Parte 3).
3. **Retropropagação** = calcular todas as culpas numa só passada para trás, reaproveitando as contas da camada seguinte (Partes 4 e 6).
4. O ciclo é sempre **frente → erro → culpa → ajuste** (Partes 4 e 7).
5. **Taxa de aprendizado, mini-batch, inicialização, ativação e normalização** decidem se o treino funciona (Partes 4, 8, 9, 11).
6. Sempre **valide em dados não vistos** e use parada antecipada (Parte 8).
7. Bibliotecas escondem o `.backward()`, mas **você sabe exatamente o que ele faz** (Parte 10).